# NB_lab_test_trends

Builds `<catalog>.gold.lab_test_trends` as defined in `sttm_silver_to_gold.xlsx`, sheet `lab_test_trends`.

* **Grain:** one row per test type (`test_name`) per calendar month
* **Driving table:** `<catalog>.silver.lab_results`, grouped by `test_name` and the year/month of `test_date`
* **Joins (LEFT):** `silver.calendar` on `date_key = date_format(date_trunc('month', test_date), 'yyyyMMdd')`
* **Window:** `LAG(abnormal_rate_pct)` by `test_name` ordered by `year, month`, for the month-over-month trend
* **Filters:** none
* **Load type:** FULL, using `CREATE OR REPLACE TABLE ... AS SELECT`

In [0]:
dbutils.widgets.text("catalog", "adb_caresync")
catalog = dbutils.widgets.get("catalog").strip()

src = f"{catalog}.silver"
target_table = f"{catalog}.gold.lab_test_trends"
print(f"Source schema: {src} | Target: {target_table}")

In [0]:
# Columns each source table must have, per the STTM column mapping
required = {
    "lab_results": ["lab_result_id", "test_name", "result_status", "test_date"],
    "calendar": ["date_key", "month_name", "quarter_name"],
}

problems = []
for tbl, cols in required.items():
    fqn = f"{src}.{tbl}"
    if not spark.catalog.tableExists(fqn):
        problems.append(f"Missing table: {fqn}")
        continue
    existing = set(spark.table(fqn).columns)
    missing = [c for c in cols if c not in existing]
    if missing:
        problems.append(f"{fqn} is missing columns: {missing}")

if problems:
    raise Exception("Source validation failed:\n" + "\n".join(problems))
print("All source tables and columns present.")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {target_table}
COMMENT 'Gold: one row per test_name per month with abnormal/critical rates and month-over-month trend (STTM lab_test_trends)'
AS
WITH monthly AS (
  SELECT
    test_name,
    year(test_date)                         AS year,
    month(test_date)                        AS month,
    COUNT(*)                                AS total_tests,
    COUNT_IF(result_status = 'Abnormal')    AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')    AS critical_test_count
  FROM {src}.lab_results
  GROUP BY test_name, year(test_date), month(test_date)
),
rates AS (
  SELECT
    *,
    CAST(ROUND(abnormal_test_count * 100.0 / NULLIF(total_tests, 0), 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(ROUND(critical_test_count * 100.0 / NULLIF(total_tests, 0), 2) AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly
),
with_lag AS (
  SELECT
    *,
    LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
  FROM rates
)
SELECT
  CAST(r.test_name AS STRING)                                         AS test_name,
  CAST(r.year AS INT)                                                 AS year,
  CAST(r.month AS INT)                                                AS month,
  c.month_name                                                        AS month_name,
  c.quarter_name                                                      AS quarter_name,
  CAST(r.total_tests AS INT)                                          AS total_tests,
  CAST(r.abnormal_test_count AS INT)                                  AS abnormal_test_count,
  CAST(r.critical_test_count AS INT)                                  AS critical_test_count,
  r.abnormal_rate_pct                                                 AS abnormal_rate_pct,
  r.critical_rate_pct                                                 AS critical_rate_pct,
  CAST(r.prev_month_abnormal_rate_pct AS DECIMAL(5,2))                AS prev_month_abnormal_rate_pct,
  CAST(r.abnormal_rate_pct - r.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM with_lag r
LEFT JOIN {src}.calendar c
  ON c.date_key = CAST(date_format(make_date(r.year, r.month, 1), 'yyyyMMdd') AS INT)
""")
print(f"Rebuilt {target_table}")

In [0]:
record_count = spark.table(target_table).count()
print(f"{target_table} record count: {record_count}")
dbutils.notebook.exit(str(record_count))